In [2]:
import numpy as np
import random as random
import matplotlib.pyplot as plt

##### Código de ejercicios anteriores

In [4]:
X_fantoma = np.array([
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.8, 0.3, 0.3, 0.3, 0.0, 0.0],
[0.0, 0.8, 0.8, 0.8, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.8, 0.3, 0.3, 0.3, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.3, 0.5, 0.3, 0.0],
[0.0, 0.3, 0.3, 0.3, 0.5, 0.5, 0.5, 0.0],
[0.0, 0.0, 0.3, 0.3, 0.3, 0.5, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
], dtype=float)

x_fantoma = X_fantoma.reshape(-1)

def mascara_rayo(n, direccion, s):

    matriz = np.array([[0] * n for _ in range(n)])      # creamos una matriz de n x n de ceros

    for i in range(0, n):      
        for j in range(0, n):
            if direccion[0] * i + direccion[1] * j == s:        # comprobamos si el píxel (i, j) pertenece al rayo
                matriz[i, j] = 1        # si pertenece al rayo, marcamos esa posición con un 1

    return matriz       # devolvemos la máscara del rayo


def matriz_proyeccion(n, direcciones):

    matriz = []     # acá vamos a guardar los vectores de todas las máscaras
    rayo = []       # acá vamos a guardar los pares de s y (alfa, beta) de cada rayo

    for direccion in direcciones:

        valores_s = []      # acá guardamos todos los valores posibles de s para la dirección actual

        # recorremos todos los píxeles
        for i in range(n):
            for j in range(n):
                valores_s.append(direccion[0] * i + direccion[1] * j)       # calculamos el valor de s que le corresponde al píxel (i, j) para esta dirección

        valores_s = sorted(set(valores_s))      # eliminamos los valores repetidos de s y los ordenamos de menor a mayor

        for s in valores_s:

            rayo.append((tuple(direccion), s))

            mascara = mascara_rayo(n, direccion, s)     # generamos la máscara de este rayo particular
            matriz.append(mascara.reshape(-1))      # convertimos la máscara de n x n en un vector y lo agregamos como una nueva fila

    return np.array(matriz) , rayo

# D2
D2 = [(1, 0), (0, 1)]
A2, rayos_A2 = matriz_proyeccion(8, D2)
rango_A2 = np.linalg.matrix_rank(A2)
nul_A2 = 64 - rango_A2


# D4
D4 = D2 + [(1, 1), (1, -1)]
A4, rayos_A4 = matriz_proyeccion(8, D4)
rango_A4 = np.linalg.matrix_rank(A4)
nul_A4 = 64 - rango_A4

# D7
D7 = D4 + [(2, 1), (1, 2), (3, 1)]
A7, rayos_A7 = matriz_proyeccion(8, D7)
rango_A7 = np.linalg.matrix_rank(A7)
nul_A7 = 64 - rango_A7


#mediciones ideales
b7 = A7@x_fantoma

#seteo la semilla
np.random.seed(18)

#z del mismo tam que b7 con componentes gaussianas estandar
z = np.random.normal(0,1, len(b7))

# tam tipico de las mediciones
escala = np.sqrt(np.mean(b7**2))

#sigmas
sigma = [0.06]
b_sigmas = {}

for s in sigma:
    # "calibramos" el ruido
    n_sigma = s*escala*z
    
    # mediciones perturbadas (limpias + ruido)
    b_sigma = b7 + n_sigma
    
    #guardamos la tupla
    b_sigmas[s] = (n_sigma, b_sigma)

    resultados = []
imagenes = []

for s in sigma:
    #traigo los datos del for de arriba
    n_sigma, b_sigma = b_sigmas[s]
    
    #obtengo la reconstruccion x_raya_sigma
    x_raya_sigma = np.linalg.lstsq(A7, b_sigma, rcond=None)[0]
    imagenes.append(x_raya_sigma)
    
    # datos para la tabla
    norma_ruido = np.linalg.norm(n_sigma)
    residuo = b_sigma - A7 @ x_raya_sigma
    norma_residuo = np.linalg.norm(residuo)
    error_relativo = np.linalg.norm(x_raya_sigma - x_fantoma) / np.linalg.norm(x_fantoma)
    valor_min = x_raya_sigma.min()
    valor_max = x_raya_sigma.max()
    
    resultados.append((s, norma_ruido, norma_residuo, error_relativo, valor_min, valor_max))


### a)

In [ ]:
residuo = b_sigma - A7 @ x_raya_sigma
norma = np.linalg.norm(A7.T @ residuo)
print(norma)

3.332735608780211e-14


Vemos que $||A^T_7r_{\sigma}|| = 3.3327{\times}e^{-14} {\approx} 0$

### b) 
**¿Por qué el resultado anterior indica que el residuo es ortogonal a todas las columnas de $A_7$ y que $A_7{\hat x}_{\sigma}$ es la proyección ortogonal de $b_{\sigma}$ sobre $Col(A_7)$?**
    Las columnas de $A_7$ son las que generan todo el $Col(A_7)$. Hacer la multiplicación de $A^T_7 {\times} r_{\sigma}$ es hacer el producto escalar de $r_{\sigma}$ contra las columnas de $A_7$. Sabemos que si el producto interno de dos vectores es $0$, son ortogonales. Como da $0$ con todas las columnas, el residuo $r_{\sigma}$ forma un ángulo de 90° con todo el subespacio $Col(A_7)$. 
    Por definicion, el único punto de un subespacio cuya distancia a un vector externo forma una recta perpendicular es la proyeccion ortogonal. Por lo tanto, $A_7{\hat x}_{\sigma}$ es la proyección ortogonal de $b_{\sigma}$ sobre $Col(A_7)$

### c) 
**¿Cuál es la diferencia entre las proyecciones tomográficas medidas por el escáner y la proyección
ortogonal que aparece en mínimos cuadrados?**
La proyección tomográfica (del escáner) es el procedimiento en el que una serie de rayos X paralelos atraviesa al paciente en un ángulo determinado y proyecta la atenuación del cuerpo sobre una línea de detectores. Transforma un corte en 2D (como se menciona en el modelo) a señales 1D.
La proyección ortogonal que aparece en mínimos cuadrados es una operación matemática que toma el vector con ruido ${b_{\sigma}}$ que no coincide con ninguna imagen y lo "baja perpendicularmente" sobre el subespacio de mediciones válidas para encontrar el vector ${A_7\hat x_\sigma}$ mas cercano posible.